# Strands Agents com AgentCore Memory (Memória de Curto Prazo)


## Introdução

Este tutorial demonstra como construir um **agente pessoal** usando Strands agents com **memória de curto prazo** (eventos brutos) do AgentCore. O agente lembra conversas recentes na sessão usando `get_last_k_turns` e pode continuar conversas de forma transparente quando o usuário retorna.


### Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial    | Curto Prazo Conversacional                                                       |
| Tipo de agente      | Agente Pessoal                                                                   |
| Framework Agêntico  | Strands Agents                                                                   |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                      |
| Componentes do tutorial | Memória de Curto Prazo AgentCore, hooks AgentInitializedEvent e MessageAddedEvent |
| Complexidade do exemplo | Iniciante                                                                    |

Você aprenderá a:
- Usar memória de curto prazo para continuidade de conversas
- Recuperar as últimas K rodadas de conversa
- Ferramenta de busca web para informações em tempo real
- Inicializar agentes com histórico de conversas

## Arquitetura
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## Pré-requisitos

- Python 3.10+
- Credenciais AWS com permissões do AgentCore Memory
- ARN da role do AgentCore Memory
- Acesso aos modelos do Amazon Bedrock

Vamos começar configurando nosso ambiente!

## Passo 1: Configuração e Imports

In [1]:
!pip install -qr requirements.txt

In [2]:
import logging
from datetime import datetime

# Setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("personal-agent")

In [3]:
# Imports
import os
from strands import Agent, tool
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent
from bedrock_agentcore.memory import MemoryClient

# Configuration
REGION = os.getenv('AWS_REGION', 'us-west-2') # AWS region for the agent
ACTOR_ID = "user_123" # It can be any unique identifier (AgentID, User ID, etc.)
SESSION_ID = "personal_session_001" # Unique session identifier


## Passo 2: Ferramenta de Busca Web

Primeiro, vamos criar uma ferramenta simples de busca web para o agente.

In [4]:
from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

@tool
def websearch(keywords: str, region: str = "us-en", max_results: int = 5) -> str:
    """Search the web for updated information.
    
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "Rate limit reached. Please try again later."
    except DDGSException as e:
        return f"Search error: {e}"
    except Exception as e:
        return f"Search error: {str(e)}"

logger.info("✅ Web search tool ready")

INFO:personal-agent:✅ Web search tool ready


## Passo 3: Criar Recurso de Memória
Para memória de curto prazo, criamos um recurso de memória sem nenhuma estratégia. Isso armazena rodadas de conversa brutas que podem ser recuperadas com `get_last_k_turns`.

In [5]:
from botocore.exceptions import ClientError

# Initialize Memory Client
client = MemoryClient(region_name=REGION)
memory_name = "PersonalAgentMemory"

try:
    # Create memory resource without strategies (thus only access to short-term memory)
    memory = client.create_memory_and_wait(
        name=memory_name,
        strategies=[],  # No strategies for short-term memory
        description="Short-term memory for personal agent",
        event_expiry_days=7, # Retention period for short-term memory. This can be upto 365 days.
    )
    memory_id = memory['id']
    logger.info(f"✅ Created memory: {memory_id}")
except ClientError as e:
    logger.info(f"❌ ERROR: {e}")
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Show any errors during memory creation
    logger.error(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"Failed to clean up memory: {cleanup_error}")

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:bedrock_agentcore.memory.client:Initialized MemoryClient for control plane: us-east-1, data plane: us-east-1
INFO:bedrock_agentcore.memory.client:Created memory: PersonalAgentMemory-gsgsQy7BYe
INFO:bedrock_agentcore.memory.client:Created memory PersonalAgentMemory-gsgsQy7BYe, waiting for ACTIVE status...
INFO:bedrock_agentcore.memory.client:Memory PersonalAgentMemory-gsgsQy7BYe is now ACTIVE (took 163 seconds)
INFO:personal-agent:✅ Created memory: PersonalAgentMemory-gsgsQy7BYe


## Passo 4: Hook de Memória

Este passo define nossa classe customizada `MemoryHookProvider` que automatiza operações de memória. Hooks são funções especiais que executam em pontos específicos do ciclo de vida de execução de um agente. O hook de memória que estamos criando serve duas funções principais:
1. **Carregar conversa recente**: Usamos o hook `AgentInitializedEvent` que carregará automaticamente o histórico de conversa recente quando o agente é inicializado.
2. **Armazenar a última mensagem**: Armazena nova mensagem conversacional.

Isso cria uma experiência de memória transparente sem gerenciamento manual.

In [6]:
class MemoryHookProvider(HookProvider):
    def __init__(self, memory_client: MemoryClient, memory_id: str):
        self.memory_client = memory_client
        self.memory_id = memory_id
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load recent conversation history when agent starts"""
        try:
            # Get session info from agent state
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")
            
            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id in agent state")
                return
            
            # Load the last 5 conversation turns from memory
            recent_turns = self.memory_client.get_last_k_turns(
                memory_id=self.memory_id,
                actor_id=actor_id,
                session_id=session_id,
                k=5
            )
            
            if recent_turns:
                # Format conversation history for context
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message['role']
                        content = message['content']['text']
                        context_messages.append(f"{role}: {content}")
                
                context = "\n".join(context_messages)
                # Add context to agent's system prompt.
                event.agent.system_prompt += f"\n\nRecent conversation:\n{context}"
                logger.info(f"✅ Loaded {len(recent_turns)} conversation turns")
                
        except Exception as e:
            logger.error(f"Memory load error: {e}")
    
    def on_message_added(self, event: MessageAddedEvent):
        """Store messages in memory"""
        messages = event.agent.messages
        try:
            # Get session info from agent state
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if messages[-1]["content"][0].get("text"):
                self.memory_client.create_event(
                    memory_id=self.memory_id,
                    actor_id=actor_id,
                    session_id=session_id,
                    messages=[(messages[-1]["content"][0]["text"], messages[-1]["role"])]
                )
        except Exception as e:
            logger.error(f"Memory save error: {e}")
    
    def register_hooks(self, registry: HookRegistry):
        # Register memory hooks
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## Passo 5: Criar Agente Pessoal com Busca Web

In [7]:
def create_personal_agent():
    """Create personal agent with memory and web search"""
    agent = Agent(
        name="PersonalAssistant",
        model="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # or your preferred model
        system_prompt=f"""You are a helpful personal assistant with web search capabilities.
        
        You can help with:
        - General questions and information lookup
        - Web searches for current information
        - Personal task management
        
        When you need current information, use the websearch function.
        Today's date: {datetime.today().strftime('%Y-%m-%d')}
        Be friendly and professional.""",
        hooks=[MemoryHookProvider(client, memory_id)],
        tools=[websearch],
        state={"actor_id": ACTOR_ID, "session_id": SESSION_ID}
    )
    return agent

# Create agent
agent = create_personal_agent()
logger.info("✅ Personal agent created with memory and web search")

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:personal-agent:✅ Personal agent created with memory and web search


#### Parabéns! Seu agente está pronto! :)
## Vamos testar o Agente

In [8]:
# Test conversation with memory
print("=== First Conversation ===")
print(f"User: My name is Alex and I'm interested in learning about AI.")
print(f"Agent: ", end="")
agent("My name is Alex and I'm interested in learning about AI.")

=== First Conversation ===
User: My name is Alex and I'm interested in learning about AI.
Agent: 

INFO:bedrock_agentcore.memory.client:Created event: 0000001774385929827#1b8b4142
INFO:strands.telemetry.metrics:Creating Strands MetricsClient


Nice to meet you, Alex! I'd be happy to help you learn about AI. That's a fascinating field with so much to explore!

There are many aspects of AI we could dive into, depending on what interests you most:

**Broad topics:**
- What is artificial intelligence and how does it work?
- Different types of AI (machine learning, deep learning, natural language processing, etc.)
- Current applications of AI in everyday life
- The history and evolution of AI
- AI ethics and responsible development

**Practical areas:**
- Getting started with AI programming and tools
- Popular AI frameworks and platforms
- Career paths in AI
- Recent breakthroughs and developments

**Specific questions:**
- How AI models like me (ChatGPT/Claude) work
- Computer vision and image recognition
- Robotics and autonomous systems
- AI in healthcare, finance, or other industries

**What would you like to start with?** 

You could also ask me specific questions about AI concepts, and I can search for the latest informatio

INFO:bedrock_agentcore.memory.client:Created event: 0000001774385933485#9a14f45a


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Nice to meet you, Alex! I'd be happy to help you learn about AI. That's a fascinating field with so much to explore!\n\nThere are many aspects of AI we could dive into, depending on what interests you most:\n\n**Broad topics:**\n- What is artificial intelligence and how does it work?\n- Different types of AI (machine learning, deep learning, natural language processing, etc.)\n- Current applications of AI in everyday life\n- The history and evolution of AI\n- AI ethics and responsible development\n\n**Practical areas:**\n- Getting started with AI programming and tools\n- Popular AI frameworks and platforms\n- Career paths in AI\n- Recent breakthroughs and developments\n\n**Specific questions:**\n- How AI models like me (ChatGPT/Claude) work\n- Computer vision and image recognition\n- Robotics and autonomous systems\n- AI in healthcare, finance, or other industries\n\n**What would you like to start w

In [9]:
print(f"User: Can you search for the latest AI trends in 2025?")
print(f"Agent: ", end="")
agent("Can you search for the latest AI trends in 2025?")

User: Can you search for the latest AI trends in 2025?
Agent: 

INFO:bedrock_agentcore.memory.client:Created event: 0000001774385936601#b28850fc



Tool #1: websearch


INFO:primp:response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=latest%20AI%20trends%202025 200
INFO:primp:response: https://grokipedia.com/api/typeahead?query=latest+AI+trends+2025&limit=1 200
INFO:primp:response: https://www.google.com/search?q=latest+AI+trends+2025&filter=1&start=0&hl=en-US&lr=lang_en&cr=countryUS 403
INFO:primp:response: https://www.mojeek.com/search?q=latest+AI+trends+2025 403
INFO:primp:response: https://html.duckduckgo.com/html/ 200


Great! Here are the **latest AI trends for 2025** based on what experts and leading tech companies are predicting:

## Key AI Trends in 2025:

**1. Agentic AI**
- AI agents that can operate more autonomously and complete complex tasks with minimal human intervention
- One of the fastest-advancing technologies according to Gartner

**2. Augmented Working**
- AI enhancing human productivity by augmenting (not replacing) workers
- AI as a collaborative tool in the workplace

**3. Real-Time Decision-Making**
- AI systems that can analyze data and make decisions instantly
- Increasingly important for business operations

**4. Generative AI Evolution**
- Continued advancement and refinement of generative AI applications
- More practical implementations across industries

**5. AI-Ready Data**
- Focus on preparing and organizing data for AI systems
- Another fastest-advancing area on Gartner's Hype Cycle

**6. Advanced AI Legislation**
- More regulations and frameworks governing AI use
- Focus

INFO:bedrock_agentcore.memory.client:Created event: 0000001774385945038#2310d86f


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Great! Here are the **latest AI trends for 2025** based on what experts and leading tech companies are predicting:\n\n## Key AI Trends in 2025:\n\n**1. Agentic AI**\n- AI agents that can operate more autonomously and complete complex tasks with minimal human intervention\n- One of the fastest-advancing technologies according to Gartner\n\n**2. Augmented Working**\n- AI enhancing human productivity by augmenting (not replacing) workers\n- AI as a collaborative tool in the workplace\n\n**3. Real-Time Decision-Making**\n- AI systems that can analyze data and make decisions instantly\n- Increasingly important for business operations\n\n**4. Generative AI Evolution**\n- Continued advancement and refinement of generative AI applications\n- More practical implementations across industries\n\n**5. AI-Ready Data**\n- Focus on preparing and organizing data for AI systems\n- Another fastest-advancing area on G

In [10]:
print(f"User: I'm particularly interested in machine learning applications.")
print(f"Agent: ", end="")
agent("I'm particularly interested in machine learning applications.")

User: I'm particularly interested in machine learning applications.
Agent: 

INFO:bedrock_agentcore.memory.client:Created event: 0000001774385945320#905080e5


Excellent choice, Alex! Machine learning is one of the most practical and impactful areas of AI. Let me search for some current information about machine learning applications in 2025.
Tool #2: websearch


INFO:bedrock_agentcore.memory.client:Created event: 0000001774385947194#34bb232a
INFO:primp:response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=machine%20learning%20applications%202025 200
INFO:primp:response: https://grokipedia.com/api/typeahead?query=machine+learning+applications+2025&limit=1 200
INFO:primp:response: https://search.brave.com/search?q=machine+learning+applications+2025&source=web 200


Perfect! Here's what's happening with **machine learning applications in 2025**:

## Current ML Applications & Trends:

**1. Fraud Detection & Financial Services**
- Traditional ML applications like credit card fraud identification continue to be crucial
- Real-time pattern recognition to prevent financial crimes
- These established systems are proving so effective that companies aren't rushing to replace them

**2. Predictive Analytics**
- Scaling predictive models to handle larger datasets
- Businesses using ML to forecast trends, customer behavior, and market changes
- Real-time decision-making based on data insights

**3. Data Processing & Insights**
- ML analyzing both structured and unstructured data
- Extracting valuable insights from massive datasets
- Improved data-driven culture in organizations

**4. Quantum Computing + ML**
- Emerging integration of quantum computing with deep learning
- Potential for exponentially faster processing of complex problems

**5. Augmented Reali

INFO:bedrock_agentcore.memory.client:Created event: 0000001774385952390#01b6327b


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Perfect! Here's what's happening with **machine learning applications in 2025**:\n\n## Current ML Applications & Trends:\n\n**1. Fraud Detection & Financial Services**\n- Traditional ML applications like credit card fraud identification continue to be crucial\n- Real-time pattern recognition to prevent financial crimes\n- These established systems are proving so effective that companies aren't rushing to replace them\n\n**2. Predictive Analytics**\n- Scaling predictive models to handle larger datasets\n- Businesses using ML to forecast trends, customer behavior, and market changes\n- Real-time decision-making based on data insights\n\n**3. Data Processing & Insights**\n- ML analyzing both structured and unstructured data\n- Extracting valuable insights from massive datasets\n- Improved data-driven culture in organizations\n\n**4. Quantum Computing + ML**\n- Emerging integration of quantum computing 

## Testar Continuidade de Memória

Para testar se nosso sistema de memória está funcionando corretamente, vamos criar uma nova instância do agente e ver se ele consegue acessar as informações previamente armazenadas:

In [11]:
# Create new agent instance (simulates user returning)
print("=== User Returns - New Session ===")
new_agent = create_personal_agent()

# Test memory continuity
print(f"User: What was my name again?")
print(f"Agent: ", end="")
new_agent("What was my name again?")

print(f"User: Can you search for more information about machine learning?")
print(f"Agent: ", end="")
new_agent("Can you search for more information about machine learning?")

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


=== User Returns - New Session ===


INFO:personal-agent:✅ Loaded 4 conversation turns


User: What was my name again?
Agent: 

INFO:bedrock_agentcore.memory.client:Created event: 0000001774385953048#26b56074


...


Tool #1: websearch


INFO:primp:response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=machine%20learning%20applications%202025%202026 200
INFO:primp:response: https://grokipedia.com/api/typeahead?query=machine+learning+applications+2025+2026&limit=1 200
INFO:primp:response: https://html.duckduckgo.com/html/ 200


Great! Here's what I found about **machine learning in 2025-2026**:

## Current Machine Learning Trends:

**1. Shift from Hype to Practical Value**
- Machine learning is moving away from theoretical applications toward real, tangible business solutions
- Companies are focusing on ROI and measurable impact

**2. Generative AI Maturation**
- Generative AI is becoming more refined and practical
- Moving beyond novelty into mainstream business applications

**3. Agentic Systems**
- Autonomous AI systems that can take independent action
- These systems are becoming increasingly capable of handling complex tasks without constant human input

**4. Small Models on Devices**
- Machine learning models are getting smaller and running directly on personal devices
- Better privacy, speed, and offline functionality
- Reducing dependence on cloud computing for every task

**5. Key Industry Use Cases**
- Predictive analytics for business forecasting
- Automation of routine tasks
- Improved business ef

INFO:bedrock_agentcore.memory.client:Created event: 0000001774385973454#27cb5e54


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Great! Here's what I found about **machine learning in 2025-2026**:\n\n## Current Machine Learning Trends:\n\n**1. Shift from Hype to Practical Value**\n- Machine learning is moving away from theoretical applications toward real, tangible business solutions\n- Companies are focusing on ROI and measurable impact\n\n**2. Generative AI Maturation**\n- Generative AI is becoming more refined and practical\n- Moving beyond novelty into mainstream business applications\n\n**3. Agentic Systems**\n- Autonomous AI systems that can take independent action\n- These systems are becoming increasingly capable of handling complex tasks without constant human input\n\n**4. Small Models on Devices**\n- Machine learning models are getting smaller and running directly on personal devices\n- Better privacy, speed, and offline functionality\n- Reducing dependence on cloud computing for every task\n\n**5. Key Industry Use

## Visualizar Memória Armazenada

In [12]:
# Check what's stored in memory
print("=== Memory Contents ===")
recent_turns = client.get_last_k_turns(
    memory_id=memory_id,
    actor_id=ACTOR_ID,
    session_id=SESSION_ID,
    k=3 # Adjust k to see more or fewer turns
)

for i, turn in enumerate(recent_turns, 1):
    print(f"Turn {i}:")
    for message in turn:
        role = message['role']
        content = message['content']['text'][:100] + "..." if len(message['content']['text']) > 100 else message['content']['text']
        print(f"  {role}: {content}")
    print()

=== Memory Contents ===
Turn 1:
  ASSISTANT: Great! Here's what I found about **machine learning in 2025-2026**:

## Current Machine Learning Tre...

Turn 2:
  USER: Can you search for more information about machine learning?
  ASSISTANT: Your name is **Alex**! You mentioned that at the beginning of our conversation when you said you wer...

Turn 3:
  USER: What was my name again?
  ASSISTANT: Perfect! Here's what's happening with **machine learning applications in 2025**:

## Current ML Appl...
  ASSISTANT: Excellent choice, Alex! Machine learning is one of the most practical and impactful areas of AI. Let...



## Resumo

Este tutorial mostrou como construir um agente pessoal. Você aprendeu:

- Criar recursos de memória sem estratégias
- Usar `get_last_k_turns` para histórico de conversas
- Adicionar capacidades de busca web aos agentes
- Implementar hooks de memória para carregamento de contexto

**Próximos Passos:**
- Adicionar ferramentas mais sofisticadas
- Implementar estratégias de memória de longo prazo
- Aprimorar capacidades de busca com múltiplas fontes

## Limpeza (Opcional)

In [13]:
# Clean up all resources created during notebook execution
print("=== Limpeza de Recursos ===")

# Step 1: Delete all events in the session (short-term memory data)
try:
    events = client.list_events(
        memory_id=memory_id,
        actor_id=ACTOR_ID,
        session_id=SESSION_ID,
        max_results=100
    )
    if events:
        logger.info(f"🗑️ Deleting {len(events)} events from session...")
        for event in events:
            event_id = event.get('eventId', event.get('id', None))
            if event_id:
                client.delete_event(
                    memory_id=memory_id,
                    actor_id=ACTOR_ID,
                    session_id=SESSION_ID,
                    event_id=event_id
                )
        logger.info(f"✅ All events deleted from session: {SESSION_ID}")
    else:
        logger.info("ℹ️ No events found in session to delete.")
except Exception as e:
    logger.warning(f"⚠️ Could not delete events (may not exist): {e}")

# Step 2: Delete the Memory resource and wait for completion
# This removes the memory resource and all associated data (actors, sessions, events)
try:
    logger.info(f"🗑️ Deleting memory resource: {memory_id}")
    client.delete_memory_and_wait(memory_id=memory_id)
    logger.info(f"✅ Memory resource fully deleted: {memory_id}")
except Exception as e:
    logger.error(f"❌ Failed to delete memory: {e}")

print("=== Limpeza concluída ===")

=== Limpeza de Recursos ===


INFO:bedrock_agentcore.memory.client:Retrieved total of 11 events
INFO:personal-agent:🗑️ Deleting 11 events from session...
INFO:personal-agent:✅ All events deleted from session: personal_session_001
INFO:personal-agent:🗑️ Deleting memory resource: PersonalAgentMemory-gsgsQy7BYe
INFO:bedrock_agentcore.memory.client:Deleted memory: PersonalAgentMemory-gsgsQy7BYe
INFO:bedrock_agentcore.memory.client:Initiated deletion of memory PersonalAgentMemory-gsgsQy7BYe
INFO:bedrock_agentcore.memory.client:Memory PersonalAgentMemory-gsgsQy7BYe successfully deleted (took 30 seconds)
INFO:personal-agent:✅ Memory resource fully deleted: PersonalAgentMemory-gsgsQy7BYe


=== Limpeza concluída ===
